In [3]:
import pandas as pd
import numpy as np
from functools import reduce

In [4]:
# Pre-processing WEF_GCI

wef_df = pd.read_csv("data/WEF_GCI.csv")

wef_df = wef_df[wef_df["INDICATOR_LABEL"].isin(["GCI 4.0: Environment-related treaties in force",
                                                "GCI 4.0: Energy efficiency regulation",
                                                "GCI 4.0: Renewable energy regulation"])]

wef_df = wef_df[wef_df["UNIT_MEASURE"] == "SCORE"]

wef_df = wef_df[["REF_AREA", "TIME_PERIOD", "INDICATOR_LABEL", "OBS_VALUE"]]

wef_df = wef_df.pivot_table(index=["REF_AREA", "TIME_PERIOD"],
                            columns="INDICATOR_LABEL",
                            values="OBS_VALUE",
                            aggfunc="first").reset_index()

wef_df["TIME_PERIOD"] = wef_df["TIME_PERIOD"].astype(int)

wef_df.columns.name = None

wef_df.to_excel("WEF.xlsx")

wef_df


,REF_AREA,TIME_PERIOD,GCI 4.0: Energy efficiency regulation,GCI 4.0: Environment-related treaties in force,GCI 4.0: Renewable energy regulation
0,AGO,2019,10.62,62.07,30.00
1,ALB,2019,71.40,82.76,69.61
2,ARE,2019,65.23,72.41,72.29
3,ARG,2019,34.08,79.31,59.00
4,ARM,2019,44.31,65.52,65.71
...,...,...,...,...,...
136,VNM,2019,72.00,79.31,66.71
137,YEM,2019,10.46,65.52,20.00
138,ZAF,2019,76.15,82.76,76.14
139,ZMB,2019,16.08,65.52,51.43


In [5]:
# Pre-processing CCPI

START_YEAR = 2010
END_YEAR = 2024

ccpi_df = pd.read_csv("data/CCPI.csv")

ccpi_df = ccpi_df.melt(id_vars="ISO",
                       value_vars=[str(year) for year in range(START_YEAR, END_YEAR+1)],
                       var_name="TIME_PERIOD",
                       value_name="CCPI_SCORE").rename(columns={"ISO": "REF_AREA"})

ccpi_df["TIME_PERIOD"] = ccpi_df["TIME_PERIOD"].astype(int)

ccpi_df.to_excel("CCPI.xlsx")

ccpi_df

,REF_AREA,TIME_PERIOD,CCPI_SCORE
0,ARG,2010,52.20
1,AUT,2010,48.20
2,AUS,2010,41.90
3,BEL,2010,57.20
4,BGR,2010,47.50
...,...,...,...
820,TUR,2024,43.82
821,TWN,2024,36.94
822,USA,2024,42.79
823,ZAF,2024,49.53


In [6]:
# Pre-processing EPS

START_YEAR = 2000
END_YEAR = 2020

eps_df = pd.read_csv("data/EPS.csv")
#eps_df = eps_df[eps_df["TIME_PERIOD"].isin(range(START_YEAR,END_YEAR+1))]
eps_df = eps_df[eps_df["CLIM_POL"]=="EPS"]
eps_df = eps_df[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]

eps_df["TIME_PERIOD"] = eps_df["TIME_PERIOD"].astype(int)

eps_df.to_excel("EPS.xlsx")

eps_df

,REF_AREA,TIME_PERIOD,OBS_VALUE
8504,CHL,1990,0.000000
8505,CHL,1991,0.000000
8506,CHL,1992,0.000000
8507,CHL,1993,0.000000
8508,CHL,1994,0.000000
...,...,...,...
21075,ZAF,2016,0.861111
21076,ZAF,2017,0.861111
21077,ZAF,2018,0.916667
21078,ZAF,2019,0.861111


In [ ]:
# Pre-processing AAMNE

aamne_df = pd.read_csv("data/AAMNE.csv")

#aamne_df = aamne_df[aamne_df["cou"].isin(eps_df["REF_AREA"].unique())]

aamne_df = aamne_df.pivot_table(index=["cou", "year"],
                                  columns=["isic", "own"],
                                  values="IMP",
                                  aggfunc="first")

aamne_df.columns = [f"{industry}_{'Domestic' if own == 'D' else 'Foreign'}"
                    for industry, own in aamne_df.columns]

aamne_df = aamne_df.reset_index().rename(columns={"cou": "REF_AREA", "year": "TIME_PERIOD"})

aamne_df["TIME_PERIOD"] = aamne_df["TIME_PERIOD"].astype(int)

aamne_df.to_excel("AAMNE.xlsx")

aamne_df

,REF_AREA,TIME_PERIOD,A01T03-Domestic,A01T03-Foreign,B05T09-Domestic,B05T09-Foreign,C10T12-Domestic,C10T12-Foreign,C13T15-Domestic,C13T15-Foreign,...,P85-Domestic,P85-Foreign,Q86T88-Domestic,Q86T88-Foreign,R90T93-Domestic,R90T93-Foreign,S94T96-Domestic,S94T96-Foreign,T97T98-Domestic,T97T98-Foreign
0,ARG,2000,740.146242,7.051148,550.584599,254.875400,704.333954,150.420595,374.100773,40.763719,...,85.559858,0.104439,235.821418,0.431290,220.601088,14.057199,127.096333,1.916266,0.0,0.0
1,ARG,2001,629.622104,5.986626,424.875925,200.751611,633.567422,129.060498,325.257072,34.325443,...,76.501373,0.093126,225.382890,0.583505,198.741700,12.060566,120.876519,1.685878,0.0,0.0
2,ARG,2002,460.011559,4.027853,279.033701,116.589509,386.616195,79.451248,186.311998,21.648745,...,31.927865,0.048138,118.392738,0.577681,93.822007,6.790805,64.445045,1.023663,0.0,0.0
3,ARG,2003,588.161519,5.428584,276.788732,125.534808,486.085035,103.664143,291.054760,32.730168,...,46.603350,0.075451,175.819929,0.852096,120.159681,9.566806,92.503238,1.467263,0.0,0.0
4,ARG,2004,692.587860,6.555194,218.355263,115.936673,733.044983,155.146757,544.714461,58.302265,...,98.161488,0.147908,298.130981,1.343515,173.265261,13.460092,136.922603,2.044794,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,ZAF,2016,2717.513184,44.311815,1835.179802,1790.295387,1135.943605,518.407786,644.105558,82.373067,...,681.932102,1.013727,1541.764083,10.749168,324.519368,9.588371,164.012588,41.615374,0.0,0.0
1613,ZAF,2017,2808.245905,44.459091,2578.777886,1981.266690,1593.477711,610.561431,757.472176,90.901999,...,665.175720,0.971821,1633.858440,11.342515,374.523860,11.169273,175.264244,43.570912,0.0,0.0
1614,ZAF,2018,3100.783810,46.387346,3105.616682,2324.723949,2182.920792,796.939367,837.447186,97.931563,...,481.011182,0.704361,1843.537762,12.808295,351.118744,10.806484,171.793723,42.026933,0.0,0.0
1615,ZAF,2019,2907.479921,46.493250,2888.158239,2389.319449,2159.032228,858.144915,787.972845,98.730040,...,426.674117,0.633159,1825.456605,13.478801,336.180798,9.862535,177.239052,43.061298,0.0,0.0


In [8]:
# Pre-processing EPI

INDICATORS = ["BCA", "CBP", "CDA", "CDF", "CHA", "FGA", "GHN", "GTI", "GTP", "LUF", "NDA"]
FILEPATHS = [f"data/EPI/{indicator}_ind_na.csv" for indicator in INDICATORS]

START_YEAR = 1995
END_YEAR = 2024

init_data = {"REF_AREA": [], "TIME_PERIOD": []}

for each in INDICATORS:
    init_data[each] = []

epi_df = pd.DataFrame(init_data)

epi_df = pd.DataFrame()

for idx, path in enumerate(FILEPATHS):
    indicator_name = INDICATORS[idx]
    df = pd.read_csv(path)
    df = df.melt(id_vars="iso",
                 value_vars=[f"{indicator_name}.ind.{year}" for year in range(START_YEAR, END_YEAR+1)],
                 var_name="TIME_PERIOD",
                 value_name=f"{indicator_name}_VALUE").rename(columns={"iso": "REF_AREA"})
    if idx == 0:
        epi_df = df
    else:
        epi_df = pd.concat([epi_df, df], axis=1)

epi_df:pd.DataFrame = epi_df.loc[:,~epi_df.columns.duplicated()].copy() # MAGIC
epi_df["TIME_PERIOD"] = epi_df["TIME_PERIOD"].str.split(".").str[-1].astype(int)

epi_df.to_excel("EPI.xlsx")

epi_df

,REF_AREA,TIME_PERIOD,BCA_VALUE,CBP_VALUE,CDA_VALUE,CDF_VALUE,CHA_VALUE,FGA_VALUE,GHN_VALUE,GTI_VALUE,GTP_VALUE,LUF_VALUE,NDA_VALUE
0,AFG,1995,42.4,97.7,0.0,0.0,6.8,19.0,18.2,0.4,9.9,44.1,16.8
1,ALB,1995,89.8,87.6,81.2,100.0,59.8,0.0,49.0,57.5,62.6,50.5,52.4
2,DZA,1995,31.1,51.5,38.5,26.9,30.2,39.4,9.5,26.7,32.4,45.8,50.1
3,AND,1995,NaN,83.7,50.8,42.5,63.1,64.4,54.6,51.9,43.7,49.3,50.6
4,AGO,1995,30.3,88.5,0.0,0.0,0.0,0.0,10.6,0.0,2.5,49.5,67.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6595,WLF,2024,41.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6596,ESH,2024,43.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6597,YEM,2024,70.1,100.0,60.4,100.0,68.8,3.8,49.0,53.0,63.8,NaN,50.6
6598,ZMB,2024,31.3,84.0,16.8,60.0,48.9,100.0,21.6,34.2,43.4,47.8,55.4


In [9]:
START_YEAR = 1995
END_YEAR = 2024

gdp_df=pd.read_csv("data/GDP_DATA.csv")

gdp_df = gdp_df.melt("Country Code",
                      [str(year) for year in range(START_YEAR, END_YEAR+1)],
                    "TIME_PERIOD",
                    "GDP").rename(columns = {"Country Code": "REF_AREA"})
gdp_df["TIME_PERIOD"] = gdp_df["TIME_PERIOD"].astype(int)
gdp_df.to_excel("GDP.xlsx")
gdp_df

,REF_AREA,TIME_PERIOD,GDP
0,ABW,1995,16548.717390
1,AFE,1995,766.522931
2,AFG,1995,NaN
3,AFW,1995,866.731606
4,AGO,1995,404.294819
...,...,...,...
7975,XKX,2024,7023.065985
7976,YEM,2024,NaN
7977,ZAF,2024,6267.186814
7978,ZMB,2024,1187.109434


In [10]:
START_YEAR =1995
END_YEAR = 2024

trade_df = pd.read_csv("data/TRADE_OPEN.csv")
trade_df = trade_df.melt("Country Code",
                      [str(year) for year in range(START_YEAR, END_YEAR+1)],
                    "TIME_PERIOD",
                    "TRADE").rename(columns = {"Country Code": "REF_AREA"})
trade_df["TIME_PERIOD"] = trade_df["TIME_PERIOD"].astype(int)
trade_df.to_excel("TRADE.xlsx")
trade_df

,REF_AREA,TIME_PERIOD,TRADE
0,ABW,1995,171.362098
1,AFE,1995,47.436309
2,AFG,1995,NaN
3,AFW,1995,NaN
4,AGO,1995,NaN
...,...,...,...
7975,XKX,2024,113.789652
7976,YEM,2024,NaN
7977,ZAF,2024,61.646770
7978,ZMB,2024,62.534887


In [11]:
# 2. Fill missing keys with placeholders
fill_values = {'REF_AREA': 'UNKNOWN_AREA', 'TIME_PERIOD': 'UNKNOWN_TIME'}
filled_dfs = [df.fillna(fill_values) for df in [eps_df, epi_df, wef_df, ccpi_df, aamne_df, gdp_df, trade_df]]

# 3. Merge using how='outer' to ensure no rows or unique columns are dropped
final_df = reduce(
    lambda left, right: pd.merge(left, right, on=['REF_AREA', 'TIME_PERIOD'], how='outer'), 
    filled_dfs
)

# 4. Revert placeholders back to NaN
final_df[['REF_AREA', 'TIME_PERIOD']] = final_df[['REF_AREA', 'TIME_PERIOD']].replace({
    'UNKNOWN_AREA': np.nan, 
    'UNKNOWN_TIME': np.nan
})

final_df.to_excel("COMPLETE_IMP.xlsx")

final_df

,REF_AREA,TIME_PERIOD,OBS_VALUE,BCA_VALUE,CBP_VALUE,CDA_VALUE,CDF_VALUE,CHA_VALUE,FGA_VALUE,GHN_VALUE,...,Q86T88-Domestic,Q86T88-Foreign,R90T93-Domestic,R90T93-Foreign,S94T96-Domestic,S94T96-Foreign,T97T98-Domestic,T97T98-Foreign,GDP,TRADE
0,ABW,1995,NaN,52.2,77.0,26.9,0.0,13.9,NaN,45.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16548.717390,171.362098
1,ABW,1996,NaN,52.2,77.0,26.9,0.0,13.9,NaN,45.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16620.954560,175.344130
2,ABW,1997,NaN,52.2,77.0,26.9,0.0,13.9,NaN,45.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17750.009560,168.781911
3,ABW,1998,NaN,52.2,77.0,26.9,0.0,13.9,NaN,45.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18828.087060,163.267360
4,ABW,1999,NaN,52.2,77.0,26.9,0.0,13.9,NaN,45.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19216.197240,164.559014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8526,ZWE,2020,NaN,37.2,96.9,62.1,100.0,63.1,7.6,33.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2059.674454,40.178406
8527,ZWE,2021,NaN,34.7,96.9,69.8,100.0,57.5,4.4,33.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2613.605421,44.114559
8528,ZWE,2022,NaN,34.1,96.9,69.1,100.0,54.2,3.8,32.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2536.400502,55.067352
8529,ZWE,2023,NaN,34.1,96.9,69.1,100.0,54.2,3.8,32.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2195.224921,40.291261
